In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
zip_path = "/content/drive/MyDrive/archive (4).zip"

In [ ]:
!unzip -q "/content/drive/MyDrive/archive (4).zip" -d /content/

In [ ]:
import os
os.listdir('/content/')

['.config',
 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1',
 'drive',
 'sample_data']

In [ ]:
path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

In [ ]:
# =====================================================
# NEXTBEAT AI – FINAL IMPROVED TRAINING PIPELINE
# =====================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install wfdb pandas scikit-learn matplotlib seaborn torch torchmetrics tqdm -q
!pip install pandas==2.2.3 -q

import wfdb
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report)
from tqdm import tqdm
import os
import json
import warnings
import time
from datetime import datetime
warnings.filterwarnings('ignore')

print("✅ Libraries installed and imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# ========== PATHS & CONFIG ==========
data_path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"
assert os.path.exists(f"{data_path}/ptbxl_database.csv"), "Dataset not found!"

class Config:
    data_path = data_path
    save_folder = "/content/drive/MyDrive/NextBeatAI_Results"
    experiment_name = f"nextbeat_improved_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    sampling_rate = 100
    seq_len = 1000
    num_leads = 12

    # Model
    ecg_cnn_channels = [64, 128, 256]
    ecg_lstm_hidden = 128
    clinical_hidden = [64, 128]
    fusion_dim = 128
    num_classes = 5
    dropout = 0.4               # reduced from 0.5
    weight_decay = 5e-4         # reduced from 1e-3
    lr = 0.001
    batch_size = 64
    max_epochs = 80             # give more time to converge
    patience = 20               # longer patience before early stopping
    grad_clip = 1.0

    class_names = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __init__(self):
        os.makedirs(self.save_folder, exist_ok=True)
        self.model_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_best.pth")
        self.results_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_results.json")

cfg = Config()
print(f"Device: {cfg.device}")

# ========== DATA LOADING (standard PTB‑XL labels) ==========
df = pd.read_csv(f"{cfg.data_path}/ptbxl_database.csv", index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(lambda x: eval(x))

def aggregate_diagnostic(y_dict):
    # Standard hierarchy: NORM takes precedence, then specific codes
    if 'NORM' in y_dict:
        return 'NORM'
    mi_codes = ['MI','IMI','AMI','PMI','LMI','ILMI','ASMI','ALMI','INJAS','INJAL','IPLMI','IPMI']
    if any(x in y_dict for x in mi_codes):
        return 'MI'
    sttc_codes = ['STTC','STD','STE','NST_','DIG','LNGQT']
    if any(x in y_dict for x in sttc_codes):
        return 'STTC'
    cd_codes = ['CD','LAFB','LPFB','IRBBB','CRBBB','CLBBB','ILBBB','LBBB','RBBB','IVCD','WPW','PRC(S)']
    if any(x in y_dict for x in cd_codes):
        return 'CD'
    hyp_codes = ['HYP','LVH','RVH','LVHYP','RVHYP','SEHYP','LAO/LAE','RAO/RAE']
    if any(x in y_dict for x in hyp_codes):
        return 'HYP'
    return 'NORM'

df['diagnostic_class'] = df.scp_codes.apply(aggregate_diagnostic)
df = df[df.diagnostic_class.isin(cfg.class_names)].copy()
class_mapping = {cls: i for i, cls in enumerate(cfg.class_names)}
df['label'] = df.diagnostic_class.map(class_mapping)

print("Class distribution:")
print(df['diagnostic_class'].value_counts().sort_index())

# Official folds
train_df = df[df.strat_fold < 9].copy()
val_df = df[df.strat_fold == 9].copy()
test_df = df[df.strat_fold == 10].copy()

# ========== CLINICAL FEATURES (3 meaningful ones only) ==========
for split in [train_df, val_df, test_df]:
    split['sex_encoded'] = (split['sex'] == 1).astype(int)
    split['num_diagnoses'] = split.scp_codes.apply(len)

age_scaler = StandardScaler()
num_diag_scaler = StandardScaler()
train_median_age = train_df['age'].median()

train_df['age_scaled'] = age_scaler.fit_transform(train_df[['age']].fillna(train_median_age))
train_df['num_diagnoses_scaled'] = num_diag_scaler.fit_transform(train_df[['num_diagnoses']])

for split in [val_df, test_df]:
    split['age_scaled'] = age_scaler.transform(split[['age']].fillna(train_median_age))
    split['num_diagnoses_scaled'] = num_diag_scaler.transform(split[['num_diagnoses']])

clinical_cols = ['age_scaled', 'sex_encoded', 'num_diagnoses_scaled']
print(f"Clinical features: {clinical_cols}")

# ========== DATASET (mild augmentation) ==========
class PTBXLDataset(Dataset):
    def __init__(self, df, data_path, training=False):
        self.df = df.reset_index(drop=True)
        self.data_path = data_path
        self.n_samples = 1000
        self.training = training

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        record = wfdb.rdrecord(f"{self.data_path}/{row['filename_lr']}",
                               sampfrom=0, sampto=self.n_samples, physical=True)
        signals = record.p_signal.T   # (12, 1000)

        # Normalize per lead
        signals = (signals - np.mean(signals, axis=1, keepdims=True)) / (np.std(signals, axis=1, keepdims=True) + 1e-8)

        # Pad/crop
        if signals.shape[1] > self.n_samples:
            signals = signals[:, :self.n_samples]
        elif signals.shape[1] < self.n_samples:
            signals = np.pad(signals, ((0,0),(0,self.n_samples - signals.shape[1])), mode='constant')

        # Mild augmentation (training only)
        if self.training:
            # time shift ±30 ms
            shift = np.random.randint(-3, 4)
            if shift > 0:
                signals = np.pad(signals, ((0,0),(shift,0)), mode='constant')[:, :self.n_samples]
            elif shift < 0:
                signals = np.pad(signals, ((0,0),(0,-shift)), mode='constant')[:, -shift:]
            # Gaussian noise σ=0.02
            signals += 0.02 * np.random.randn(*signals.shape)

        clinical = row[clinical_cols].values.astype(np.float32)
        label = row['label']
        return (torch.tensor(signals, dtype=torch.float32),
                torch.tensor(clinical, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

# ========== DATALOADERS (simple shuffle, no sampler) ==========
train_ds = PTBXLDataset(train_df, cfg.data_path, training=True)
val_ds = PTBXLDataset(val_df, cfg.data_path, training=False)
test_ds = PTBXLDataset(test_df, cfg.data_path, training=False)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

# ========== MODEL (Cross‑Modal Attention) ==========
class ECGEncoder(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(12, 64, 5, 2, 2), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64, 128, 5, 2, 2), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(128, 256, 5, 2, 2), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
        )
        self.lstm = nn.LSTM(256, 128, batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return torch.cat((h_n[-2], h_n[-1]), dim=1)   # (batch, 256)

class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim=3, dropout=0.4):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU()
        )

    def forward(self, x): return self.mlp(x)

class CrossModalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, ecg_feat, clin_feat):
        Q = self.query(ecg_feat)
        K = self.key(clin_feat)
        V = self.value(clin_feat)
        scores = (Q * K).sum(dim=1, keepdim=True) * self.scale
        weights = torch.sigmoid(scores)
        fused = ecg_feat + weights * (V - ecg_feat)
        return fused, weights

class NextBeatAI(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ecg_encoder = ECGEncoder(dropout=cfg.dropout)
        self.clinical_encoder = ClinicalEncoder(input_dim=len(clinical_cols), dropout=cfg.dropout)
        self.attention = CrossModalAttention(cfg.fusion_dim)
        self.ecg_proj = nn.Linear(self.ecg_encoder.lstm.hidden_size*2, cfg.fusion_dim)
        self.clin_proj = nn.Linear(128, cfg.fusion_dim)
        self.classifier = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim//2),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim//2, cfg.num_classes)
        )

    def forward(self, ecg, clinical):
        ecg_feat = self.ecg_encoder(ecg)
        clin_feat = self.clinical_encoder(clinical)
        ecg_feat = self.ecg_proj(ecg_feat)
        clin_feat = self.clin_proj(clin_feat)
        fused, _ = self.attention(ecg_feat, clin_feat)
        return self.classifier(fused)

model = NextBeatAI(cfg).to(cfg.device)
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ========== CLASS‑BALANCED LOSS (proven for imbalanced data) ==========
class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes, device):
        super().__init__()
        # Effective number of samples
        effective_num = 1.0 - np.power(0.9999, samples_per_class)
        weights = (1.0 - 0.9999) / np.array(effective_num)
        weights = weights / weights.sum() * num_classes
        self.weights = torch.tensor(weights, dtype=torch.float32).to(device)

    def forward(self, logits, labels):
        return F.cross_entropy(logits, labels, weight=self.weights)

class_counts = train_df['label'].value_counts().sort_index().values
criterion = ClassBalancedLoss(class_counts, cfg.num_classes, cfg.device)

optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ========== TRAINING FUNCTIONS ==========
def train_epoch():
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    for ecg, clin, labels in tqdm(train_loader, desc="Train", leave=False):
        ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
        optimizer.zero_grad()
        logits = model(ecg, clin)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/len(train_loader), accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro')

def evaluate(loader):
    model.eval()
    total_loss, all_preds, all_labels, all_probs = 0, [], [], []
    with torch.no_grad():
        for ecg, clin, labels in loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            total_loss += criterion(logits, labels).item()
            probs = torch.softmax(logits, 1)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    try: auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except: auc = 0.0
    return total_loss/len(loader), acc, f1, auc

# ========== MAIN TRAINING LOOP ==========
print("\n🚀 Starting training (Improved Pipeline)...")
best_val_f1 = 0
patience_counter = 0
start_time = time.time()

for epoch in range(cfg.max_epochs):
    train_loss, train_acc, train_f1 = train_epoch()
    val_loss, val_acc, val_f1, val_auc = evaluate(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} F1={train_f1:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), cfg.model_save_path)
        print(f"   ✅ Best model (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= cfg.patience:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining finished in {(time.time()-start_time)/60:.1f} min")

# ========== TEST EVALUATION ==========
print("\n📊 TEST SET EVALUATION")
if os.path.exists(cfg.model_save_path):
    model.load_state_dict(torch.load(cfg.model_save_path))
    test_loss, test_acc, test_f1, test_auc = evaluate(test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print(f"Test Macro AUROC: {test_auc:.4f}")

    # Per‑class report
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ecg, clin, labels in test_loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            all_preds.extend(torch.argmax(logits,1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print("\n" + classification_report(all_labels, all_preds, target_names=cfg.class_names, digits=4))
else:
    print("❌ No saved model found.")

# Save results
results = {
    "test_accuracy": float(test_acc) if 'test_acc' in locals() else 0,
    "test_macro_f1": float(test_f1) if 'test_f1' in locals() else 0,
    "test_macro_auroc": float(test_auc) if 'test_auc' in locals() else 0,
    "best_val_f1": float(best_val_f1),
    "clinical_features_used": clinical_cols,
    "loss_function": "ClassBalancedLoss (inverse effective frequency)"
}
with open(cfg.results_save_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n📁 Results saved to {cfg.results_save_path}")
print("\n🎉 NextBeatAI Improved Pipeline complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatib

Epoch 01: Train Loss=1.3893 Acc=0.6199 F1=0.3678 | Val Loss=1.2916 Acc=0.5239 F1=0.3752 AUC=0.7954
   ✅ Best model (F1=0.3752)


Epoch 02: Train Loss=1.2194 Acc=0.6205 F1=0.4465 | Val Loss=1.1874 Acc=0.5983 F1=0.4302 AUC=0.8296
   ✅ Best model (F1=0.4302)


Epoch 03: Train Loss=1.1528 Acc=0.6341 F1=0.4734 | Val Loss=1.1212 Acc=0.5841 F1=0.4602 AUC=0.8482
   ✅ Best model (F1=0.4602)


Epoch 04: Train Loss=1.1164 Acc=0.6429 F1=0.4927 | Val Loss=1.1139 Acc=0.6334 F1=0.4794 AUC=0.8530
   ✅ Best model (F1=0.4794)


Epoch 05: Train Loss=1.0796 Acc=0.6535 F1=0.5154 | Val Loss=1.1139 Acc=0.5695 F1=0.4498 AUC=0.8567


... (54 outputs left out)

Epoch 60: Train Loss=0.6302 Acc=0.7756 F1=0.6825 | Val Loss=1.0236 Acc=0.6658 F1=0.5365 AUC=0.8877


Epoch 61: Train Loss=0.6388 Acc=0.7736 F1=0.6806 | Val Loss=1.0193 Acc=0.6708 F1=0.5396 AUC=0.8882


Epoch 62: Train Loss=0.6309 Acc=0.7789 F1=0.6871 | Val Loss=1.0204 Acc=0.6699 F1=0.5368 AUC=0.8895


Epoch 63: Train Loss=0.6260 Acc=0.7781 F1=0.6865 | Val Loss=1.0239 Acc=0.6689 F1=0.5362 AUC=0.8892
⏹️ Early stopping at epoch 63

Training finished in 78.8 min

📊 TEST SET EVALUATION
Test Accuracy: 0.6891
Test Macro F1: 0.5465
Test Macro AUROC: 0.8912

              precision    recall  f1-score   support

          CD     0.5527    0.7004    0.6179       247
         HYP     0.3289    0.4153    0.3670       118
          MI     0.7978    0.6637    0.7246       553
        NORM     0.8813    0.7351    0.8016      1212
        STTC     0.1407    0.5205    0.2216        73

    accuracy                         0.6891      2203
   macro avg     0.5403    0.6070    0.5465      2203
weighted avg     0.7694    0.6891    0.7192      2203


📁 Results saved to /content/drive/MyDrive/NextBeatAI_Results/nextbeat_improved_20260503_164625_results.json

🎉 NextBeatAI Improved Pipeline complete!


In [ ]:
# =====================================================
# NEXTBEAT AI – FINAL IMPROVED TRAINING PIPELINE
# =====================================================

# 1. Mount Google Drive (so we can read the dataset and save results)
from google.colab import drive
drive.mount('/content/drive')   # Colab's way to access files stored in your Drive

# 2. Install missing Python packages (only if not already installed)
#    - wfdb: to read ECG files in WFDB format
#    - pandas, scikit-learn, etc.: for data handling and metrics
#    - torch, torchmetrics: for deep learning
#    - tqdm: for progress bars
!pip install wfdb pandas scikit-learn matplotlib seaborn torch torchmetrics tqdm -q
# Force a specific pandas version to avoid conflicts (this is a quick fix)
!pip install pandas==2.2.3 -q

# 3. Import all necessary libraries
import wfdb                # to load ECG records
import pandas as pd        # for tabular data (DataFrames)
import numpy as np         # for numerical operations
import torch               # PyTorch: main deep learning framework
import torch.nn as nn      # neural network layers
import torch.nn.functional as F  # functions like softmax, cross-entropy
import torch.optim as optim      # optimizers (Adam, SGD, …)
from torch.utils.data import Dataset, DataLoader  # to create datasets and load in batches
from sklearn.preprocessing import StandardScaler  # to normalise clinical features
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report)  # evaluation metrics
from tqdm import tqdm       # progress bar for training loops
import os                   # file and path operations
import json                 # to save results as JSON
import warnings             # to suppress annoying warnings
import time                 # to measure training time
from datetime import datetime  # to timestamp the experiment
warnings.filterwarnings('ignore')   # keep the output clean

print("✅ Libraries installed and imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")  # will be True if GPU is present

# ========== PATHS & CONFIG ==========
# The folder where the PTB‑XL dataset is stored (after you have uploaded it to Colab)
data_path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"
# Make sure the main CSV file exists – if not, the program stops.
assert os.path.exists(f"{data_path}/ptbxl_database.csv"), "Dataset not found!"

# A configuration class to hold all hyper‑parameters and paths
class Config:
    data_path = data_path
    # Where to save the final results (inside your Google Drive)
    save_folder = "/content/drive/MyDrive/NextBeatAI_Results"
    # A unique experiment name with timestamp
    experiment_name = f"nextbeat_improved_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Data processing parameters
    sampling_rate = 100      # the dataset is provided at 100 Hz, we keep that
    seq_len = 1000           # we take 1000 time steps (10 seconds) per recording
    num_leads = 12           # standard 12‑lead ECG

    # Model architecture sizes
    ecg_cnn_channels = [64, 128, 256]   # number of filters in each CNN layer
    ecg_lstm_hidden = 128               # LSTM hidden units per direction
    clinical_hidden = [64, 128]         # hidden layer sizes for clinical MLP
    fusion_dim = 128                    # dimension after projecting ECG and clinical features
    num_classes = 5                     # we have 5 diagnostic classes
    dropout = 0.4                       # dropout probability (fraction of neurons to drop)
    weight_decay = 5e-4                 # L2 regularisation strength (penalty on large weights)
    lr = 0.001                          # learning rate for Adam
    batch_size = 64                     # number of samples per batch
    max_epochs = 80                     # maximum number of training epochs
    patience = 20                       # early stopping: stop if validation F1 doesn't improve for 20 epochs
    grad_clip = 1.0                     # clip gradients to this norm to avoid exploding gradients

    # Names of the target classes (in the order they appear)
    class_names = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
    # Use GPU if available, otherwise CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __init__(self):
        # Create the save folder if it doesn't exist
        os.makedirs(self.save_folder, exist_ok=True)
        # Full path for the best model weights
        self.model_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_best.pth")
        # Full path for the final results (JSON)
        self.results_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_results.json")

# Instantiate the config
cfg = Config()
print(f"Device: {cfg.device}")

# ========== DATA LOADING (standard PTB‑XL labels) ==========
# Read the main CSV file that contains all ECG filenames and patient metadata
df = pd.read_csv(f"{cfg.data_path}/ptbxl_database.csv", index_col='ecg_id')
# The 'scp_codes' column stores a dictionary of diagnostic codes – we convert it from string to a real dict
df.scp_codes = df.scp_codes.apply(lambda x: eval(x))

# This function will map the set of SCP codes to one of our 5 classes.
# It follows a hierarchy: NORM (normal) is preferred; if not, check for MI, then STTC, then CD, then HYP.
def aggregate_diagnostic(y_dict):
    # If 'NORM' is present, it's a normal ECG
    if 'NORM' in y_dict:
        return 'NORM'
    # Check for myocardial infarction codes
    mi_codes = ['MI','IMI','AMI','PMI','LMI','ILMI','ASMI','ALMI','INJAS','INJAL','IPLMI','IPMI']
    if any(x in y_dict for x in mi_codes):
        return 'MI'
    # Check for ST/T changes
    sttc_codes = ['STTC','STD','STE','NST_','DIG','LNGQT']
    if any(x in y_dict for x in sttc_codes):
        return 'STTC'
    # Check for conduction disturbances
    cd_codes = ['CD','LAFB','LPFB','IRBBB','CRBBB','CLBBB','ILBBB','LBBB','RBBB','IVCD','WPW','PRC(S)']
    if any(x in y_dict for x in cd_codes):
        return 'CD'
    # Check for hypertrophy
    hyp_codes = ['HYP','LVH','RVH','LVHYP','RVHYP','SEHYP','LAO/LAE','RAO/RAE']
    if any(x in y_dict for x in hyp_codes):
        return 'HYP'
    # If nothing matched, treat as normal (this is a safety fallback)
    return 'NORM'

# Apply the function to create a new column 'diagnostic_class'
df['diagnostic_class'] = df.scp_codes.apply(aggregate_diagnostic)
# Keep only rows whose class is one of our 5 target classes (discard any other rare classes)
df = df[df.diagnostic_class.isin(cfg.class_names)].copy()
# Create a mapping from class name to integer (0..4) for the neural network
class_mapping = {cls: i for i, cls in enumerate(cfg.class_names)}
df['label'] = df.diagnostic_class.map(class_mapping)

# Print the number of samples per class to see the imbalance
print("Class distribution:")
print(df['diagnostic_class'].value_counts().sort_index())

# The PTB‑XL dataset already provides stratified folds (0‑9 for training, 9 for validation, 10 for test)
# We use folds 0‑8 for training, fold 9 for validation, fold 10 for testing.
train_df = df[df.strat_fold < 9].copy()
val_df = df[df.strat_fold == 9].copy()
test_df = df[df.strat_fold == 10].copy()

# ========== CLINICAL FEATURES (3 meaningful ones only) ==========
# We will use: age (scaled), sex (0/1), and the number of diagnoses (scaled)
for split in [train_df, val_df, test_df]:
    # Sex: 1 = male, 0 = female (we map to integer)
    split['sex_encoded'] = (split['sex'] == 1).astype(int)
    # Number of distinct SCP codes in the record (as a proxy for complexity)
    split['num_diagnoses'] = split.scp_codes.apply(len)

# Create scalers for age and num_diagnoses so they have mean 0 and std 1
age_scaler = StandardScaler()
num_diag_scaler = StandardScaler()
# For age, we fill missing values with the median age from training set (robust)
train_median_age = train_df['age'].median()

# Fit the scalers on the training set only (to avoid data leakage)
train_df['age_scaled'] = age_scaler.fit_transform(train_df[['age']].fillna(train_median_age))
train_df['num_diagnoses_scaled'] = num_diag_scaler.fit_transform(train_df[['num_diagnoses']])

# Transform validation and test sets using the training scalers
for split in [val_df, test_df]:
    split['age_scaled'] = age_scaler.transform(split[['age']].fillna(train_median_age))
    split['num_diagnoses_scaled'] = num_diag_scaler.transform(split[['num_diagnoses']])

# The final list of clinical features we will feed into the network
clinical_cols = ['age_scaled', 'sex_encoded', 'num_diagnoses_scaled']
print(f"Clinical features: {clinical_cols}")

# ========== DATASET CLASS (handles loading ECG signals on the fly) ==========
class PTBXLDataset(Dataset):
    def __init__(self, df, data_path, training=False):
        self.df = df.reset_index(drop=True)   # we don't need the original index
        self.data_path = data_path
        self.n_samples = 1000                 # fixed length for each ECG
        self.training = training              # whether to apply augmentation

    def __len__(self):
        return len(self.df)   # number of samples in this dataset

    def __getitem__(self, idx):
        # Get the row corresponding to this index
        row = self.df.iloc[idx]
        # Load the ECG recording from the WFDB files (we load only first 1000 samples)
        record = wfdb.rdrecord(f"{self.data_path}/{row['filename_lr']}",
                               sampfrom=0, sampto=self.n_samples, physical=True)
        # p_signal is a numpy array of shape (time, leads). We transpose to (leads, time)
        signals = record.p_signal.T   # now (12, 1000)

        # Normalise each lead independently to zero mean and unit variance
        # This makes the signals comparable regardless of lead-specific amplitude.
        signals = (signals - np.mean(signals, axis=1, keepdims=True)) / (np.std(signals, axis=1, keepdims=True) + 1e-8)

        # In case the record is shorter than 1000 samples, pad with zeros.
        # If longer, we crop to 1000.
        if signals.shape[1] > self.n_samples:
            signals = signals[:, :self.n_samples]
        elif signals.shape[1] < self.n_samples:
            signals = np.pad(signals, ((0,0),(0,self.n_samples - signals.shape[1])), mode='constant')

        # Mild data augmentation (only during training)
        if self.training:
            # Random time shift of ±3 samples (~±30ms) to make the model invariant to small misalignments
            shift = np.random.randint(-3, 4)
            if shift > 0:
                signals = np.pad(signals, ((0,0),(shift,0)), mode='constant')[:, :self.n_samples]
            elif shift < 0:
                signals = np.pad(signals, ((0,0),(0,-shift)), mode='constant')[:, -shift:]
            # Add very small Gaussian noise to increase robustness (like adding jitter)
            signals += 0.02 * np.random.randn(*signals.shape)

        # Retrieve the clinical features for this sample (they are already scaled)
        clinical = row[clinical_cols].values.astype(np.float32)
        # The integer label (0‑4)
        label = row['label']
        # Return everything as PyTorch tensors
        return (torch.tensor(signals, dtype=torch.float32),
                torch.tensor(clinical, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

# ========== CREATE DATALOADERS ==========
# Each dataset object loads data on the fly; the DataLoader batches them and shuffles.
train_ds = PTBXLDataset(train_df, cfg.data_path, training=True)
val_ds = PTBXLDataset(val_df, cfg.data_path, training=False)
test_ds = PTBXLDataset(test_df, cfg.data_path, training=False)

# DataLoader takes care of feeding batches: 
# - shuffle=True for training to randomise order
# - num_workers=2 loads data in parallel to speed up
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

# ========== MODEL DEFINITION ==========
# ---- ECG Encoder: 1D CNN + Bidirectional LSTM ----
class ECGEncoder(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        # A sequence of 1D convolutional layers.
        # We use stride=2 to downsample the time dimension, and kernel_size=5.
        # The channel count goes: 12 (leads) -> 64 -> 128 -> 256.
        self.cnn = nn.Sequential(
            nn.Conv1d(12, 64, 5, 2, 2),   # in_channels=12, out_channels=64, kernel=5, stride=2, padding=2
            nn.BatchNorm1d(64),           # normalise activations within batch for stable training
            nn.ReLU(),                    # non‑linearity: max(0,x)
            nn.Dropout(dropout),          # randomly zero out some neurons to prevent overfitting
            nn.Conv1d(64, 128, 5, 2, 2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 256, 5, 2, 2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # Bidirectional LSTM to capture temporal context in both directions.
        # input_size = 256 (from CNN output), hidden_size = 128, bidirectional=True gives 128*2 = 256 output features.
        self.lstm = nn.LSTM(256, 128, batch_first=True, bidirectional=True)

    def forward(self, x):
        # x shape: (batch, 12, 1000)
        x = self.cnn(x)   # shape: (batch, 256, 125) after downsampling (1000 -> 500 -> 250 -> 125)
        # LSTM expects (batch, seq_len, features), so we swap dimensions
        x = x.permute(0, 2, 1)   # now (batch, 125, 256)
        # Run the LSTM; we only need the final hidden states (h_n) from both directions
        _, (h_n, _) = self.lstm(x)   # h_n shape: (2 * num_layers, batch, hidden) = (2, batch, 128)
        # Concatenate the forward and backward final hidden states
        # h_n[-2] is the forward last layer, h_n[-1] is the backward last layer.
        return torch.cat((h_n[-2], h_n[-1]), dim=1)   # (batch, 256)

# ---- Clinical Encoder: simple MLP ----
class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim=3, dropout=0.4):
        super().__init__()
        # Two linear layers with BatchNorm, ReLU, and Dropout in between
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.mlp(x)   # (batch, 128)

# ---- Cross‑Modal Attention (fuses ECG and clinical features) ----
class CrossModalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Linear layers for query, key, and value
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5   # 1/sqrt(dim) for scaling the dot product

    def forward(self, ecg_feat, clin_feat):
        # ecg_feat: (batch, dim), clin_feat: (batch, dim)
        Q = self.query(ecg_feat)          # (batch, dim)
        K = self.key(clin_feat)           # (batch, dim)
        V = self.value(clin_feat)         # (batch, dim)
        # Compute a single scalar attention score as the summed element‑wise product, then scaled
        scores = (Q * K).sum(dim=1, keepdim=True) * self.scale   # (batch, 1)
        # Squeeze the score through a sigmoid to get a gate in (0,1)
        weights = torch.sigmoid(scores)   # (batch, 1)
        # Residual fusion: ecg_feat + weight * (V - ecg_feat)
        # If weight=0, output is pure ECG; if weight=1, output is pure clinical projection (V)
        fused = ecg_feat + weights * (V - ecg_feat)
        return fused, weights   # we return weights only for debugging (not used later)

# ---- Full Model ----
class NextBeatAI(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ecg_encoder = ECGEncoder(dropout=cfg.dropout)
        self.clinical_encoder = ClinicalEncoder(input_dim=len(clinical_cols), dropout=cfg.dropout)
        self.attention = CrossModalAttention(cfg.fusion_dim)
        # Project ECG output (256) to fusion_dim (128)
        self.ecg_proj = nn.Linear(self.ecg_encoder.lstm.hidden_size*2, cfg.fusion_dim)
        # Project clinical output (128) to fusion_dim (128)
        self.clin_proj = nn.Linear(128, cfg.fusion_dim)
        # Final classifier: 2‑layer MLP with ReLU and Dropout
        self.classifier = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim//2),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim//2, cfg.num_classes)
        )

    def forward(self, ecg, clinical):
        # Encode both modalities
        ecg_feat = self.ecg_encoder(ecg)          # (batch, 256)
        clin_feat = self.clinical_encoder(clinical)  # (batch, 128)
        # Project both to the same fusion space
        ecg_feat = self.ecg_proj(ecg_feat)        # (batch, 128)
        clin_feat = self.clin_proj(clin_feat)     # (batch, 128)
        # Fuse via attention
        fused, _ = self.attention(ecg_feat, clin_feat)  # (batch, 128)
        # Classify
        return self.classifier(fused)             # (batch, 5) logits

# Create the model and move it to the appropriate device (GPU/CPU)
model = NextBeatAI(cfg).to(cfg.device)
# Print the number of trainable parameters
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ========== CLASS‑BALANCED LOSS (to handle imbalanced data) ==========
class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes, device):
        super().__init__()
        # Compute effective number of samples for each class
        # Effective number = (1 - beta^n) / (1 - beta), with beta=0.9999
        beta = 0.9999
        effective_num = 1.0 - np.power(beta, samples_per_class)   # array per class
        # The raw weight = (1 - beta) / effective_num
        weights = (1.0 - beta) / effective_num
        # Normalise so that sum(weights) = num_classes (so loss magnitude doesn't change)
        weights = weights / weights.sum() * num_classes
        # Convert to torch tensor and move to device
        self.weights = torch.tensor(weights, dtype=torch.float32).to(device)

    def forward(self, logits, labels):
        # Use PyTorch's cross entropy with the given class weights
        # This computes: - weights[label] * log(softmax(logits)[label])
        return F.cross_entropy(logits, labels, weight=self.weights)

# Get the number of training samples per class (in the same order as class names)
class_counts = train_df['label'].value_counts().sort_index().values
criterion = ClassBalancedLoss(class_counts, cfg.num_classes, cfg.device)

# ========== OPTIMISER AND SCHEDULER ==========
# Adam optimiser: adaptive learning rates per parameter
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
# ReduceLROnPlateau: if validation loss doesn't decrease for 5 epochs, multiply learning rate by 0.5
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ========== TRAINING AND EVALUATION FUNCTIONS ==========
def train_epoch():
    """Run one full training epoch and return average loss, accuracy, and macro F1."""
    model.train()   # set model to training mode (enables dropout, batch norm uses batch stats)
    total_loss = 0
    all_preds = []
    all_labels = []
    # Loop over batches from the training loader
    for ecg, clin, labels in tqdm(train_loader, desc="Train", leave=False):
        # Move data to GPU if available
        ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
        # Zero the gradients from the previous iteration
        optimizer.zero_grad()
        # Forward pass: get logits from the model
        logits = model(ecg, clin)
        # Compute the class‑balanced loss
        loss = criterion(logits, labels)
        # Backward pass: compute gradients
        loss.backward()
        # Clip gradients to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        # Update model weights
        optimizer.step()
        # Accumulate loss
        total_loss += loss.item()
        # Store predictions and true labels for metrics
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    # Compute metrics
    avg_loss = total_loss / len(train_loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1

def evaluate(loader):
    """Evaluate the model on a given data loader (validation or test)."""
    model.eval()   # set to evaluation mode (dropout disabled, batch norm uses running stats)
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []   # to compute AUROC
    # No gradient needed for evaluation (saves memory and computation)
    with torch.no_grad():
        for ecg, clin, labels in loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, 1)   # convert logits to probabilities
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    # Compute AUROC (one‑vs‑rest) for multi‑class
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    return total_loss/len(loader), acc, f1, auc

# ========== MAIN TRAINING LOOP ==========
print("\n🚀 Starting training (Improved Pipeline)...")
best_val_f1 = 0
patience_counter = 0
start_time = time.time()

for epoch in range(cfg.max_epochs):
    # Train for one epoch
    train_loss, train_acc, train_f1 = train_epoch()
    # Evaluate on validation set
    val_loss, val_acc, val_f1, val_auc = evaluate(val_loader)
    # Update learning rate based on validation loss
    scheduler.step(val_loss)

    # Print progress
    print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} F1={train_f1:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")

    # Save model if validation F1 improves
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), cfg.model_save_path)
        print(f"   ✅ Best model (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        # Early stopping: if no improvement for 'patience' epochs, stop training
        if patience_counter >= cfg.patience:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining finished in {(time.time()-start_time)/60:.1f} min")

# ========== FINAL TEST EVALUATION ==========
print("\n📊 TEST SET EVALUATION")
# Load the best model (the one with highest validation F1)
if os.path.exists(cfg.model_save_path):
    model.load_state_dict(torch.load(cfg.model_save_path))
    test_loss, test_acc, test_f1, test_auc = evaluate(test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print(f"Test Macro AUROC: {test_auc:.4f}")

    # Per‑class detailed report
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ecg, clin, labels in test_loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print("\n" + classification_report(all_labels, all_preds, target_names=cfg.class_names, digits=4))
else:
    print("❌ No saved model found.")

# Save the final results as a JSON file (for record keeping)
results = {
    "test_accuracy": float(test_acc) if 'test_acc' in locals() else 0,
    "test_macro_f1": float(test_f1) if 'test_f1' in locals() else 0,
    "test_macro_auroc": float(test_auc) if 'test_auc' in locals() else 0,
    "best_val_f1": float(best_val_f1),
    "clinical_features_used": clinical_cols,
    "loss_function": "ClassBalancedLoss (inverse effective frequency)"
}
with open(cfg.results_save_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n📁 Results saved to {cfg.results_save_path}")
print("\n🎉 NextBeatAI Improved Pipeline complete!")

# =====================================================
# NEXTBEAT AI – FINAL IMPROVED TRAINING PIPELINE
# =====================================================

# 1. Mount Google Drive (so we can read the dataset and save results)
from google.colab import drive
drive.mount('/content/drive')   # Colab's way to access files stored in your Drive

# 2. Install missing Python packages (only if not already installed)
#    - wfdb: to read ECG files in WFDB format
#    - pandas, scikit-learn, etc.: for data handling and metrics
#    - torch, torchmetrics: for deep learning
#    - tqdm: for progress bars
!pip install wfdb pandas scikit-learn matplotlib seaborn torch torchmetrics tqdm -q
# Force a specific pandas version to avoid conflicts (this is a quick fix)
!pip install pandas==2.2.3 -q

# 3. Import all necessary libraries
import wfdb                # to load ECG records
import pandas as pd        # for tabular data (DataFrames)
import numpy as np         # for numerical operations
import torch               # PyTorch: main deep learning framework
import torch.nn as nn      # neural network layers
import torch.nn.functional as F  # functions like softmax, cross-entropy
import torch.optim as optim      # optimizers (Adam, SGD, …)
from torch.utils.data import Dataset, DataLoader  # to create datasets and load in batches
from sklearn.preprocessing import StandardScaler  # to normalise clinical features
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report)  # evaluation metrics
from tqdm import tqdm       # progress bar for training loops
import os                   # file and path operations
import json                 # to save results as JSON
import warnings             # to suppress annoying warnings
import time                 # to measure training time
from datetime import datetime  # to timestamp the experiment
warnings.filterwarnings('ignore')   # keep the output clean

print("✅ Libraries installed and imported")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")  # will be True if GPU is present

# ========== PATHS & CONFIG ==========
# The folder where the PTB‑XL dataset is stored (after you have uploaded it to Colab)
data_path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"
# Make sure the main CSV file exists – if not, the program stops.
assert os.path.exists(f"{data_path}/ptbxl_database.csv"), "Dataset not found!"

# A configuration class to hold all hyper‑parameters and paths
class Config:
    data_path = data_path
    # Where to save the final results (inside your Google Drive)
    save_folder = "/content/drive/MyDrive/NextBeatAI_Results"
    # A unique experiment name with timestamp
    experiment_name = f"nextbeat_improved_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

    # Data processing parameters
    sampling_rate = 100      # the dataset is provided at 100 Hz, we keep that
    seq_len = 1000           # we take 1000 time steps (10 seconds) per recording
    num_leads = 12           # standard 12‑lead ECG

    # Model architecture sizes
    ecg_cnn_channels = [64, 128, 256]   # number of filters in each CNN layer
    ecg_lstm_hidden = 128               # LSTM hidden units per direction
    clinical_hidden = [64, 128]         # hidden layer sizes for clinical MLP
    fusion_dim = 128                    # dimension after projecting ECG and clinical features
    num_classes = 5                     # we have 5 diagnostic classes
    dropout = 0.4                       # dropout probability (fraction of neurons to drop)
    weight_decay = 5e-4                 # L2 regularisation strength (penalty on large weights)
    lr = 0.001                          # learning rate for Adam
    batch_size = 64                     # number of samples per batch
    max_epochs = 80                     # maximum number of training epochs
    patience = 20                       # early stopping: stop if validation F1 doesn't improve for 20 epochs
    grad_clip = 1.0                     # clip gradients to this norm to avoid exploding gradients

    # Names of the target classes (in the order they appear)
    class_names = ['CD', 'HYP', 'MI', 'NORM', 'STTC']
    # Use GPU if available, otherwise CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def __init__(self):
        # Create the save folder if it doesn't exist
        os.makedirs(self.save_folder, exist_ok=True)
        # Full path for the best model weights
        self.model_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_best.pth")
        # Full path for the final results (JSON)
        self.results_save_path = os.path.join(self.save_folder, f"{self.experiment_name}_results.json")

# Instantiate the config
cfg = Config()
print(f"Device: {cfg.device}")

# ========== DATA LOADING (standard PTB‑XL labels) ==========
# Read the main CSV file that contains all ECG filenames and patient metadata
df = pd.read_csv(f"{cfg.data_path}/ptbxl_database.csv", index_col='ecg_id')
# The 'scp_codes' column stores a dictionary of diagnostic codes – we convert it from string to a real dict
df.scp_codes = df.scp_codes.apply(lambda x: eval(x))

# This function will map the set of SCP codes to one of our 5 classes.
# It follows a hierarchy: NORM (normal) is preferred; if not, check for MI, then STTC, then CD, then HYP.
def aggregate_diagnostic(y_dict):
    # If 'NORM' is present, it's a normal ECG
    if 'NORM' in y_dict:
        return 'NORM'
    # Check for myocardial infarction codes
    mi_codes = ['MI','IMI','AMI','PMI','LMI','ILMI','ASMI','ALMI','INJAS','INJAL','IPLMI','IPMI']
    if any(x in y_dict for x in mi_codes):
        return 'MI'
    # Check for ST/T changes
    sttc_codes = ['STTC','STD','STE','NST_','DIG','LNGQT']
    if any(x in y_dict for x in sttc_codes):
        return 'STTC'
    # Check for conduction disturbances
    cd_codes = ['CD','LAFB','LPFB','IRBBB','CRBBB','CLBBB','ILBBB','LBBB','RBBB','IVCD','WPW','PRC(S)']
    if any(x in y_dict for x in cd_codes):
        return 'CD'
    # Check for hypertrophy
    hyp_codes = ['HYP','LVH','RVH','LVHYP','RVHYP','SEHYP','LAO/LAE','RAO/RAE']
    if any(x in y_dict for x in hyp_codes):
        return 'HYP'
    # If nothing matched, treat as normal (this is a safety fallback)
    return 'NORM'

# Apply the function to create a new column 'diagnostic_class'
df['diagnostic_class'] = df.scp_codes.apply(aggregate_diagnostic)
# Keep only rows whose class is one of our 5 target classes (discard any other rare classes)
df = df[df.diagnostic_class.isin(cfg.class_names)].copy()
# Create a mapping from class name to integer (0..4) for the neural network
class_mapping = {cls: i for i, cls in enumerate(cfg.class_names)}
df['label'] = df.diagnostic_class.map(class_mapping)

# Print the number of samples per class to see the imbalance
print("Class distribution:")
print(df['diagnostic_class'].value_counts().sort_index())

# The PTB‑XL dataset already provides stratified folds (0‑9 for training, 9 for validation, 10 for test)
# We use folds 0‑8 for training, fold 9 for validation, fold 10 for testing.
train_df = df[df.strat_fold < 9].copy()
val_df = df[df.strat_fold == 9].copy()
test_df = df[df.strat_fold == 10].copy()

# ========== CLINICAL FEATURES (3 meaningful ones only) ==========
# We will use: age (scaled), sex (0/1), and the number of diagnoses (scaled)
for split in [train_df, val_df, test_df]:
    # Sex: 1 = male, 0 = female (we map to integer)
    split['sex_encoded'] = (split['sex'] == 1).astype(int)
    # Number of distinct SCP codes in the record (as a proxy for complexity)
    split['num_diagnoses'] = split.scp_codes.apply(len)

# Create scalers for age and num_diagnoses so they have mean 0 and std 1
age_scaler = StandardScaler()
num_diag_scaler = StandardScaler()
# For age, we fill missing values with the median age from training set (robust)
train_median_age = train_df['age'].median()

# Fit the scalers on the training set only (to avoid data leakage)
train_df['age_scaled'] = age_scaler.fit_transform(train_df[['age']].fillna(train_median_age))
train_df['num_diagnoses_scaled'] = num_diag_scaler.fit_transform(train_df[['num_diagnoses']])

# Transform validation and test sets using the training scalers
for split in [val_df, test_df]:
    split['age_scaled'] = age_scaler.transform(split[['age']].fillna(train_median_age))
    split['num_diagnoses_scaled'] = num_diag_scaler.transform(split[['num_diagnoses']])

# The final list of clinical features we will feed into the network
clinical_cols = ['age_scaled', 'sex_encoded', 'num_diagnoses_scaled']
print(f"Clinical features: {clinical_cols}")

# ========== DATASET CLASS (handles loading ECG signals on the fly) ==========
class PTBXLDataset(Dataset):
    def __init__(self, df, data_path, training=False):
        self.df = df.reset_index(drop=True)   # we don't need the original index
        self.data_path = data_path
        self.n_samples = 1000                 # fixed length for each ECG
        self.training = training              # whether to apply augmentation

    def __len__(self):
        return len(self.df)   # number of samples in this dataset

    def __getitem__(self, idx):
        # Get the row corresponding to this index
        row = self.df.iloc[idx]
        # Load the ECG recording from the WFDB files (we load only first 1000 samples)
        record = wfdb.rdrecord(f"{self.data_path}/{row['filename_lr']}",
                               sampfrom=0, sampto=self.n_samples, physical=True)
        # p_signal is a numpy array of shape (time, leads). We transpose to (leads, time)
        signals = record.p_signal.T   # now (12, 1000)

        # Normalise each lead independently to zero mean and unit variance
        # This makes the signals comparable regardless of lead-specific amplitude.
        signals = (signals - np.mean(signals, axis=1, keepdims=True)) / (np.std(signals, axis=1, keepdims=True) + 1e-8)

        # In case the record is shorter than 1000 samples, pad with zeros.
        # If longer, we crop to 1000.
        if signals.shape[1] > self.n_samples:
            signals = signals[:, :self.n_samples]
        elif signals.shape[1] < self.n_samples:
            signals = np.pad(signals, ((0,0),(0,self.n_samples - signals.shape[1])), mode='constant')

        # Mild data augmentation (only during training)
        if self.training:
            # Random time shift of ±3 samples (~±30ms) to make the model invariant to small misalignments
            shift = np.random.randint(-3, 4)
            if shift > 0:
                signals = np.pad(signals, ((0,0),(shift,0)), mode='constant')[:, :self.n_samples]
            elif shift < 0:
                signals = np.pad(signals, ((0,0),(0,-shift)), mode='constant')[:, -shift:]
            # Add very small Gaussian noise to increase robustness (like adding jitter)
            signals += 0.02 * np.random.randn(*signals.shape)

        # Retrieve the clinical features for this sample (they are already scaled)
        clinical = row[clinical_cols].values.astype(np.float32)
        # The integer label (0‑4)
        label = row['label']
        # Return everything as PyTorch tensors
        return (torch.tensor(signals, dtype=torch.float32),
                torch.tensor(clinical, dtype=torch.float32),
                torch.tensor(label, dtype=torch.long))

# ========== CREATE DATALOADERS ==========
# Each dataset object loads data on the fly; the DataLoader batches them and shuffles.
train_ds = PTBXLDataset(train_df, cfg.data_path, training=True)
val_ds = PTBXLDataset(val_df, cfg.data_path, training=False)
test_ds = PTBXLDataset(test_df, cfg.data_path, training=False)

# DataLoader takes care of feeding batches: 
# - shuffle=True for training to randomise order
# - num_workers=2 loads data in parallel to speed up
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

# ========== MODEL DEFINITION ==========
# ---- ECG Encoder: 1D CNN + Bidirectional LSTM ----
class ECGEncoder(nn.Module):
    def __init__(self, dropout=0.4):
        super().__init__()
        # A sequence of 1D convolutional layers.
        # We use stride=2 to downsample the time dimension, and kernel_size=5.
        # The channel count goes: 12 (leads) -> 64 -> 128 -> 256.
        self.cnn = nn.Sequential(
            nn.Conv1d(12, 64, 5, 2, 2),   # in_channels=12, out_channels=64, kernel=5, stride=2, padding=2
            nn.BatchNorm1d(64),           # normalise activations within batch for stable training
            nn.ReLU(),                    # non‑linearity: max(0,x)
            nn.Dropout(dropout),          # randomly zero out some neurons to prevent overfitting
            nn.Conv1d(64, 128, 5, 2, 2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(128, 256, 5, 2, 2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # Bidirectional LSTM to capture temporal context in both directions.
        # input_size = 256 (from CNN output), hidden_size = 128, bidirectional=True gives 128*2 = 256 output features.
        self.lstm = nn.LSTM(256, 128, batch_first=True, bidirectional=True)

    def forward(self, x):
        # x shape: (batch, 12, 1000)
        x = self.cnn(x)   # shape: (batch, 256, 125) after downsampling (1000 -> 500 -> 250 -> 125)
        # LSTM expects (batch, seq_len, features), so we swap dimensions
        x = x.permute(0, 2, 1)   # now (batch, 125, 256)
        # Run the LSTM; we only need the final hidden states (h_n) from both directions
        _, (h_n, _) = self.lstm(x)   # h_n shape: (2 * num_layers, batch, hidden) = (2, batch, 128)
        # Concatenate the forward and backward final hidden states
        # h_n[-2] is the forward last layer, h_n[-1] is the backward last layer.
        return torch.cat((h_n[-2], h_n[-1]), dim=1)   # (batch, 256)

# ---- Clinical Encoder: simple MLP ----
class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim=3, dropout=0.4):
        super().__init__()
        # Two linear layers with BatchNorm, ReLU, and Dropout in between
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.mlp(x)   # (batch, 128)

# ---- Cross‑Modal Attention (fuses ECG and clinical features) ----
class CrossModalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Linear layers for query, key, and value
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** -0.5   # 1/sqrt(dim) for scaling the dot product

    def forward(self, ecg_feat, clin_feat):
        # ecg_feat: (batch, dim), clin_feat: (batch, dim)
        Q = self.query(ecg_feat)          # (batch, dim)
        K = self.key(clin_feat)           # (batch, dim)
        V = self.value(clin_feat)         # (batch, dim)
        # Compute a single scalar attention score as the summed element‑wise product, then scaled
        scores = (Q * K).sum(dim=1, keepdim=True) * self.scale   # (batch, 1)
        # Squeeze the score through a sigmoid to get a gate in (0,1)
        weights = torch.sigmoid(scores)   # (batch, 1)
        # Residual fusion: ecg_feat + weight * (V - ecg_feat)
        # If weight=0, output is pure ECG; if weight=1, output is pure clinical projection (V)
        fused = ecg_feat + weights * (V - ecg_feat)
        return fused, weights   # we return weights only for debugging (not used later)

# ---- Full Model ----
class NextBeatAI(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ecg_encoder = ECGEncoder(dropout=cfg.dropout)
        self.clinical_encoder = ClinicalEncoder(input_dim=len(clinical_cols), dropout=cfg.dropout)
        self.attention = CrossModalAttention(cfg.fusion_dim)
        # Project ECG output (256) to fusion_dim (128)
        self.ecg_proj = nn.Linear(self.ecg_encoder.lstm.hidden_size*2, cfg.fusion_dim)
        # Project clinical output (128) to fusion_dim (128)
        self.clin_proj = nn.Linear(128, cfg.fusion_dim)
        # Final classifier: 2‑layer MLP with ReLU and Dropout
        self.classifier = nn.Sequential(
            nn.Linear(cfg.fusion_dim, cfg.fusion_dim//2),
            nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.fusion_dim//2, cfg.num_classes)
        )

    def forward(self, ecg, clinical):
        # Encode both modalities
        ecg_feat = self.ecg_encoder(ecg)          # (batch, 256)
        clin_feat = self.clinical_encoder(clinical)  # (batch, 128)
        # Project both to the same fusion space
        ecg_feat = self.ecg_proj(ecg_feat)        # (batch, 128)
        clin_feat = self.clin_proj(clin_feat)     # (batch, 128)
        # Fuse via attention
        fused, _ = self.attention(ecg_feat, clin_feat)  # (batch, 128)
        # Classify
        return self.classifier(fused)             # (batch, 5) logits

# Create the model and move it to the appropriate device (GPU/CPU)
model = NextBeatAI(cfg).to(cfg.device)
# Print the number of trainable parameters
print(f"Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# ========== CLASS‑BALANCED LOSS (to handle imbalanced data) ==========
class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes, device):
        super().__init__()
        # Compute effective number of samples for each class
        # Effective number = (1 - beta^n) / (1 - beta), with beta=0.9999
        beta = 0.9999
        effective_num = 1.0 - np.power(beta, samples_per_class)   # array per class
        # The raw weight = (1 - beta) / effective_num
        weights = (1.0 - beta) / effective_num
        # Normalise so that sum(weights) = num_classes (so loss magnitude doesn't change)
        weights = weights / weights.sum() * num_classes
        # Convert to torch tensor and move to device
        self.weights = torch.tensor(weights, dtype=torch.float32).to(device)

    def forward(self, logits, labels):
        # Use PyTorch's cross entropy with the given class weights
        # This computes: - weights[label] * log(softmax(logits)[label])
        return F.cross_entropy(logits, labels, weight=self.weights)

# Get the number of training samples per class (in the same order as class names)
class_counts = train_df['label'].value_counts().sort_index().values
criterion = ClassBalancedLoss(class_counts, cfg.num_classes, cfg.device)

# ========== OPTIMISER AND SCHEDULER ==========
# Adam optimiser: adaptive learning rates per parameter
optimizer = optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
# ReduceLROnPlateau: if validation loss doesn't decrease for 5 epochs, multiply learning rate by 0.5
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ========== TRAINING AND EVALUATION FUNCTIONS ==========
def train_epoch():
    """Run one full training epoch and return average loss, accuracy, and macro F1."""
    model.train()   # set model to training mode (enables dropout, batch norm uses batch stats)
    total_loss = 0
    all_preds = []
    all_labels = []
    # Loop over batches from the training loader
    for ecg, clin, labels in tqdm(train_loader, desc="Train", leave=False):
        # Move data to GPU if available
        ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
        # Zero the gradients from the previous iteration
        optimizer.zero_grad()
        # Forward pass: get logits from the model
        logits = model(ecg, clin)
        # Compute the class‑balanced loss
        loss = criterion(logits, labels)
        # Backward pass: compute gradients
        loss.backward()
        # Clip gradients to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        # Update model weights
        optimizer.step()
        # Accumulate loss
        total_loss += loss.item()
        # Store predictions and true labels for metrics
        all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    # Compute metrics
    avg_loss = total_loss / len(train_loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1

def evaluate(loader):
    """Evaluate the model on a given data loader (validation or test)."""
    model.eval()   # set to evaluation mode (dropout disabled, batch norm uses running stats)
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []   # to compute AUROC
    # No gradient needed for evaluation (saves memory and computation)
    with torch.no_grad():
        for ecg, clin, labels in loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.softmax(logits, 1)   # convert logits to probabilities
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    all_probs = np.vstack(all_probs)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    # Compute AUROC (one‑vs‑rest) for multi‑class
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    return total_loss/len(loader), acc, f1, auc

# ========== MAIN TRAINING LOOP ==========
print("\n🚀 Starting training (Improved Pipeline)...")
best_val_f1 = 0
patience_counter = 0
start_time = time.time()

for epoch in range(cfg.max_epochs):
    # Train for one epoch
    train_loss, train_acc, train_f1 = train_epoch()
    # Evaluate on validation set
    val_loss, val_acc, val_f1, val_auc = evaluate(val_loader)
    # Update learning rate based on validation loss
    scheduler.step(val_loss)

    # Print progress
    print(f"Epoch {epoch+1:02d}: Train Loss={train_loss:.4f} Acc={train_acc:.4f} F1={train_f1:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} F1={val_f1:.4f} AUC={val_auc:.4f}")

    # Save model if validation F1 improves
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), cfg.model_save_path)
        print(f"   ✅ Best model (F1={val_f1:.4f})")
    else:
        patience_counter += 1
        # Early stopping: if no improvement for 'patience' epochs, stop training
        if patience_counter >= cfg.patience:
            print(f"⏹️ Early stopping at epoch {epoch+1}")
            break

print(f"\nTraining finished in {(time.time()-start_time)/60:.1f} min")

# ========== FINAL TEST EVALUATION ==========
print("\n📊 TEST SET EVALUATION")
# Load the best model (the one with highest validation F1)
if os.path.exists(cfg.model_save_path):
    model.load_state_dict(torch.load(cfg.model_save_path))
    test_loss, test_acc, test_f1, test_auc = evaluate(test_loader)
    print(f"Test Accuracy: {test_acc:.4f}")
    print(f"Test Macro F1: {test_f1:.4f}")
    print(f"Test Macro AUROC: {test_auc:.4f}")

    # Per‑class detailed report
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for ecg, clin, labels in test_loader:
            ecg, clin, labels = ecg.to(cfg.device), clin.to(cfg.device), labels.to(cfg.device)
            logits = model(ecg, clin)
            all_preds.extend(torch.argmax(logits, 1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    print("\n" + classification_report(all_labels, all_preds, target_names=cfg.class_names, digits=4))
else:
    print("❌ No saved model found.")

# Save the final results as a JSON file (for record keeping)
results = {
    "test_accuracy": float(test_acc) if 'test_acc' in locals() else 0,
    "test_macro_f1": float(test_f1) if 'test_f1' in locals() else 0,
    "test_macro_auroc": float(test_auc) if 'test_auc' in locals() else 0,
    "best_val_f1": float(best_val_f1),
    "clinical_features_used": clinical_cols,
    "loss_function": "ClassBalancedLoss (inverse effective frequency)"
}
with open(cfg.results_save_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"\n📁 Results saved to {cfg.results_save_path}")
print("\n🎉 NextBeatAI Improved Pipeline complete!")